# Kinase Drug Discovery — Notebook 5
## Ligand-based modelling: QSAR

**Learning goals**

1. Retrieve and **curate** kinase bioactivity data from ChEMBL (the step that determines model quality more than any algorithm).
2. Build and **validate** a QSAR model for a single kinase (EGFR) — and see how much the validation strategy (random vs scaffold vs temporal split) changes the apparent performance.
3. Understand **y-scrambling** and the **applicability domain**.

Data for all 12 kinases are downloaded and curated here; the curated set (`data/kinase_dataset_curated.csv.gz`) is reused in **Notebook 6 (proteochemometrics)**, so run at least sections 1–2 of this notebook first.

**Hardware / runtime** (4 CPU cores, no GPU, < 45 GB RAM)

| Step | Time (approx.) |
|---|---|
| First-time ChEMBL download of 12 kinases (cached afterwards) | 5–15 min, depends on the ChEMBL server |
| Curation and standardisation | 2–4 min |
| QSAR section | 5 min |

Peak memory stays below ~4 GB. All heavy steps are cached in `data/` so re-running is fast.

## 0. Setup and configuration

In [ ]:
import os
N_CPU = min(4, os.cpu_count() or 1)
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_v, str(N_CPU))

import time, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from tqdm.auto import tqdm
from joblib import Parallel, delayed

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (r2_score, mean_squared_error, roc_auc_score, matthews_corrcoef,
                             balanced_accuracy_score)
from scipy.stats import pearsonr, spearmanr

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.width", 160)

DATA = Path("data"); DATA.mkdir(exist_ok=True)
SEED = 42
rng = np.random.default_rng(SEED)
print("Using", N_CPU, "CPU threads")

In [ ]:
# ----------------------------- configuration -----------------------------
# 12 human kinases: ChEMBL single-protein target IDs
TARGETS = {
    "EGFR":  "CHEMBL203",  "ERBB2": "CHEMBL1824", "ABL1":  "CHEMBL1862", "SRC":   "CHEMBL267",
    "LCK":   "CHEMBL258",  "BTK":   "CHEMBL5251", "KIT":   "CHEMBL1936", "KDR":   "CHEMBL279",
    "FLT3":  "CHEMBL1974", "JAK2":  "CHEMBL2971", "AURKA": "CHEMBL4722", "CDK2":  "CHEMBL301",
}
ACTIVITY_TYPES = ["IC50", "Ki", "Kd", "EC50"]
ACTIVE_THRESHOLD = 6.5     # pChEMBL >= 6.5 (~316 nM) counts as "active" (Papyrus convention)
QSAR_TARGET = "EGFR"
N_TREES = 200

## 1. Retrieving bioactivity data from ChEMBL

We use the ChEMBL REST API and request only activities with a **pChEMBL value** (= −log10 of a molar IC50/Ki/Kd/EC50, standardised by ChEMBL). Results are cached as `data/chembl_<ID>.csv.gz`. The public API occasionally returns *500 Internal Server Error* for large requests; the code then automatically retries with simpler, smaller requests, and an interrupted download resumes where it stopped (`data/chembl_<ID>.partial.json`). The IC50/Ki/Kd/EC50 filter is applied locally.

> **Working offline / when the ChEMBL API is down.** If you have a local ChEMBL database, export the result of the query below to `data/chembl_36_kinases.csv` (comma- or tab-separated, with a header). The next cell uses that file automatically and does not contact the API.
>
> ```sql
> SELECT md.chembl_id AS molecule_chembl_id, cs.canonical_smiles AS canonical_smiles,
>        act.pchembl_value AS pchembl_value, act.standard_type AS standard_type,
>        act.standard_relation AS standard_relation, a.chembl_id AS assay_chembl_id,
>        a.assay_type AS assay_type, td.chembl_id AS target_chembl_id,
>        act.data_validity_comment AS data_validity_comment, act.potential_duplicate AS potential_duplicate,
>        d.chembl_id AS document_chembl_id, d.year AS document_year
> FROM chembl_36.activities act
> JOIN chembl_36.assays a              ON a.assay_id  = act.assay_id
> JOIN chembl_36.target_dictionary td  ON td.tid      = a.tid
> JOIN chembl_36.molecule_dictionary md ON md.molregno = act.molregno
> LEFT JOIN chembl_36.compound_structures cs ON cs.molregno = act.molregno
> LEFT JOIN chembl_36.docs d           ON d.doc_id    = act.doc_id
> WHERE td.chembl_id IN ('CHEMBL203','CHEMBL1824','CHEMBL1862','CHEMBL267','CHEMBL258','CHEMBL5251',
>                        'CHEMBL1936','CHEMBL279','CHEMBL1974','CHEMBL2971','CHEMBL4722','CHEMBL301')
>   AND act.pchembl_value IS NOT NULL
>   AND act.standard_type IN ('IC50','Ki','Kd','EC50');
> ```

> **Tip — Papyrus.** For larger studies, use the **Papyrus** dataset (Béquignon *et al.*, J Cheminform 2023; `pip install papyrus-scripts`), which aggregates and standardises ChEMBL, ExCAPE-DB and other sources with quality labels. The ChEMBL API is used here because it requires no large download.

In [ ]:
CHEMBL_API = "https://www.ebi.ac.uk/chembl/api/data/activity.json"
FIELDS = ["molecule_chembl_id", "canonical_smiles", "pchembl_value", "standard_type", "standard_relation",
          "assay_chembl_id", "assay_type", "target_chembl_id", "data_validity_comment",
          "potential_duplicate", "document_chembl_id", "document_year"]

# The ChEMBL server sometimes answers large or field-restricted queries with "500 Internal Server Error"
# (time-outs on its side). We therefore fall back to progressively simpler/smaller requests:
#   (page size, use the 'only=' field selection)
STRATEGIES = [(1000, True), (1000, False), (500, False), (200, False), (100, False)]
ECHO_URLS = True        # print every URL before it is sent (copy-paste it into a browser or curl to check)
session = requests.Session()
session.headers.update({"Accept": "application/json", "User-Agent": "kinase-course-notebook"})

class ChEMBLError(RuntimeError):
    pass

def get_page(target_id, offset, level, attempts_per_level=3):
    """Fetch one page starting at `offset`; on server errors step down to simpler requests.
    Returns (json, level_used)."""
    while level < len(STRATEGIES):
        page_size, use_only = STRATEGIES[level]
        params = {"target_chembl_id": target_id, "pchembl_value__isnull": "false",
                  "limit": page_size, "offset": offset}
        if use_only:
            params["only"] = ",".join(FIELDS)
        for attempt in range(1, attempts_per_level + 1):
            try:
                prepared = session.prepare_request(requests.Request("GET", CHEMBL_API, params=params))
                if ECHO_URLS:
                    print(f"  GET {prepared.url}", flush=True)
                r = session.send(prepared, timeout=180)
                if r.status_code == 429:                      # rate limited: wait and retry
                    time.sleep(int(r.headers.get("Retry-After", 30)))
                    continue
                r.raise_for_status()
                return r.json(), level
            except (requests.HTTPError, requests.ConnectionError, requests.Timeout, ValueError) as exc:
                resp = getattr(exc, "response", None)
                code = getattr(resp, "status_code", None)
                if resp is not None and resp.text:
                    print(f"  server said: {resp.text[:300]!r}")
                print(f"  {target_id} offset {offset}: {type(exc).__name__} {code or ''} "
                      f"(page size {page_size}, only={use_only}, attempt {attempt})")
                if code is not None and 400 <= code < 500:
                    break                                     # client error: retrying the same request won't help
                time.sleep(5 * attempt)
        level += 1
        if level < len(STRATEGIES):
            print(f"  -> switching to simpler request: page size {STRATEGIES[level][0]}, only={STRATEGIES[level][1]}")
    raise ChEMBLError(f"ChEMBL API keeps failing for {target_id} at offset {offset}. "
                      "Check https://www.ebi.ac.uk/chembl/ (maintenance?) and re-run this cell later — "
                      "pages already downloaded are kept and the download resumes where it stopped.")

def fetch_chembl_activities(target_id):
    """All activities with a pChEMBL value for one target. Cached in data/; resumable after an interruption."""
    cache = DATA / f"chembl_{target_id}.csv.gz"
    if cache.exists():
        return pd.read_csv(cache)
    partial = DATA / f"chembl_{target_id}.partial.json"
    state = json.loads(partial.read_text()) if partial.exists() else {"offset": 0, "level": 0, "rows": []}
    rows, offset, level, total = state["rows"], state["offset"], state["level"], None
    if offset:
        print(f"{target_id}: resuming at record {offset}")
    with tqdm(desc=target_id, unit=" records", initial=offset) as bar:
        while total is None or offset < total:
            page, level = get_page(target_id, offset, level)
            acts = page.get("activities", [])
            total = page["page_meta"]["total_count"]
            bar.total = total
            if not acts:
                break
            rows.extend({k: a.get(k) for k in FIELDS} for a in acts)
            offset += len(acts)
            bar.update(len(acts))
            partial.write_text(json.dumps({"offset": offset, "level": level, "rows": rows}))
            time.sleep(0.2)                                   # be polite to the public server
    df = pd.DataFrame(rows).reindex(columns=FIELDS)
    df = df[df.standard_type.isin(ACTIVITY_TYPES)]          # IC50/Ki/Kd/EC50 filter applied locally
    df.to_csv(cache, index=False)
    partial.unlink(missing_ok=True)
    return df

# ---- Option A: a CSV exported from a local ChEMBL database (see the SQL query in the markdown above) ----
LOCAL_CSV = DATA / "chembl_36_kinases.csv"     # also accepts .csv.gz / .tsv; set to None to always use the API

def load_local_csv(path):
    """Read the SQL export, normalise column names/NULLs and map ChEMBL target IDs to gene names."""
    path = Path(path)
    opener = __import__("gzip").open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8", errors="replace") as fh:
        header = fh.readline()
    sep = max(["\t", ",", ";", "|"], key=header.count)          # detect the delimiter from the header
    df = pd.read_csv(path, sep=sep, na_values=["NULL", "null", "\\N", "None"], low_memory=False)
    # 'chembl_36.activities.pchembl_value', '"PCHEMBL_VALUE"' ... -> 'pchembl_value'
    df.columns = [str(c).strip().strip('"`\'').split(".")[-1].lower() for c in df.columns]
    missing = [c for c in FIELDS if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name} lacks the columns {missing}; found {list(df.columns)}. "
                         "Use the aliases from the SQL query (AS molecule_chembl_id, ...).")
    df["pchembl_value"] = pd.to_numeric(df.pchembl_value, errors="coerce")
    df = df[df.standard_type.isin(ACTIVITY_TYPES)]
    df["gene"] = df.target_chembl_id.map({tid: gene for gene, tid in TARGETS.items()})
    if df.gene.isna().any():
        print(f"ignoring {df.gene.isna().sum()} rows for targets not in TARGETS")
    absent = [g for g, tid in TARGETS.items() if tid not in set(df.target_chembl_id)]
    if absent:
        print("WARNING: no rows in the CSV for", absent)
    return df.dropna(subset=["gene"])[["gene"] + FIELDS].reset_index(drop=True)

# ---- Option B: the ChEMBL REST API (used when no local CSV is present) ----
if LOCAL_CSV is not None and Path(LOCAL_CSV).exists():
    print(f"Loading local ChEMBL export: {LOCAL_CSV}")
    raw = load_local_csv(LOCAL_CSV)
else:
    print("No local CSV found -> downloading from the ChEMBL web API")
    raw = pd.concat({gene: fetch_chembl_activities(tid) for gene, tid in TARGETS.items()},
                    names=["gene", None]).reset_index(level=0).reset_index(drop=True)
print(raw.shape)
raw.groupby("gene").size().rename("raw records").to_frame().T

**Troubleshooting the download.** If the cell above fails, run the diagnostic cell below. It sends three increasingly specific requests and echoes each URL, so you can paste them into a browser or `curl` to see which part the server rejects. If even the status request fails, ChEMBL is down or blocked by your network or proxy.

In [ ]:
def probe(url, params=None):
    prepared = session.prepare_request(requests.Request("GET", url, params=params))
    print("GET", prepared.url)
    try:
        r = session.send(prepared, timeout=60)
        print("   ->", r.status_code, r.reason, "|", r.text[:200].replace("\n", " "), "\n")
    except Exception as exc:
        print("   ->", type(exc).__name__, exc, "\n")

probe("https://www.ebi.ac.uk/chembl/api/data/status.json")
probe(CHEMBL_API, {"target_chembl_id": "CHEMBL203", "limit": 1})
probe(CHEMBL_API, {"target_chembl_id": "CHEMBL203", "pchembl_value__isnull": "false", "limit": 1})
probe(CHEMBL_API, {"target_chembl_id": "CHEMBL203", "pchembl_value__isnull": "false", "limit": 1,
                   "only": ",".join(FIELDS)})

## 2. Data curation

Typical curation steps (cf. Fourches *et al.*, J Chem Inf Model 2010; Papyrus):

1. keep exact measurements (`standard_relation == '='`) — censored values (">10 µM") are not usable for regression;
2. keep **binding assays** (`assay_type == 'B'`);
3. drop records flagged by ChEMBL curators (`data_validity_comment`) and **potential duplicates** (the same value reported again in a later paper);
4. **standardise** structures: remove salts/solvents (largest fragment), neutralise charges, canonical SMILES;
5. **aggregate** multiple measurements of the same compound–target pair (median), and record their spread.

In [ ]:
df = raw.dropna(subset=["canonical_smiles", "pchembl_value"]).copy()
steps = [("all records with pChEMBL", len(df))]
df = df[df.standard_relation == "="];                              steps.append(("relation '='", len(df)))
df = df[df.assay_type == "B"];                                     steps.append(("binding assays", len(df)))
df = df[df.data_validity_comment.isna()];                          steps.append(("no validity flag", len(df)))
df = df[~df.potential_duplicate.astype(str).str.lower().isin(["1", "true", "1.0"])]
steps.append(("not a potential duplicate", len(df)))
pd.DataFrame(steps, columns=["filter", "records"])

In [ ]:
def standardize_smiles(smi):
    """Cleanup -> largest fragment -> neutralise -> canonical SMILES (None on failure)."""
    try:
        m = Chem.MolFromSmiles(smi)
        if m is None:
            return None
        m = rdMolStandardize.Cleanup(m)
        m = rdMolStandardize.LargestFragmentChooser().choose(m)
        m = rdMolStandardize.Uncharger().uncharge(m)
        return Chem.MolToSmiles(m)
    except Exception:
        return None

std_cache = DATA / "standardized_smiles.json"
std_map = json.loads(std_cache.read_text()) if std_cache.exists() else {}
todo = [s for s in df.canonical_smiles.unique() if s not in std_map]
if todo:
    out = Parallel(n_jobs=N_CPU, batch_size=256)(delayed(standardize_smiles)(s) for s in tqdm(todo, desc="standardising"))
    std_map.update(dict(zip(todo, out)))
    std_cache.write_text(json.dumps(std_map))
df["smiles"] = df.canonical_smiles.map(std_map)
df = df.dropna(subset=["smiles"])
print(f"{df.smiles.nunique()} unique standardised compounds")

In [ ]:
agg = (df.groupby(["gene", "smiles"])
         .agg(pchembl=("pchembl_value", "median"),
              pchembl_std=("pchembl_value", "std"),
              n_measurements=("pchembl_value", "size"),
              year=("document_year", "min"),
              chembl_id=("molecule_chembl_id", "first"))
         .reset_index())
agg["active"] = (agg.pchembl >= ACTIVE_THRESHOLD).astype(int)
agg.to_csv(DATA / "kinase_dataset_curated.csv.gz", index=False)

overview = agg.groupby("gene").agg(compounds=("smiles", "size"), median_pchembl=("pchembl", "median"),
                                   frac_active=("active", "mean")).round(2)
overview.sort_values("compounds", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
order = overview.sort_values("compounds", ascending=False).index
sns.violinplot(data=agg, x="gene", y="pchembl", order=order, ax=axes[0], cut=0, color="lightsteelblue")
axes[0].axhline(ACTIVE_THRESHOLD, ls="--", c="crimson"); axes[0].set_title("pChEMBL distribution per kinase")
axes[0].tick_params(axis="x", rotation=45)
rep = agg.dropna(subset=["pchembl_std"])
axes[1].hist(rep.pchembl_std, bins=50, color="grey")
axes[1].set_title(f"Spread of repeated measurements (n={len(rep)} pairs)\nmedian SD = {rep.pchembl_std.median():.2f} log units")
axes[1].set_xlabel("SD of pChEMBL for the same compound–kinase pair")
plt.tight_layout(); plt.show()

The right-hand panel is important: the **experimental uncertainty** of public pChEMBL data is typically ~0.5 log units (Kramer *et al.*, J Med Chem 2012). A model with RMSE ≈ 0.6–0.7 is therefore already close to the noise level — no model can be "better than the data".

## 3. QSAR for a single kinase (EGFR)

### 3.1 Descriptors

We encode molecules as **Morgan fingerprints** (radius 2, 2048 bits; ECFP4-like). Each bit indicates the presence of a circular substructure.

In [ ]:
def fingerprints(smiles, radius=2, n_bits=2048):
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    mols = [Chem.MolFromSmiles(s) for s in smiles]
    X = np.vstack([gen.GetFingerprintAsNumPy(m) for m in mols]).astype(np.uint8)
    bitvects = [gen.GetFingerprint(m) for m in mols]
    return X, bitvects

def regression_metrics(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "Pearson r": pearsonr(y_true, y_pred)[0] if np.std(y_pred) > 0 else np.nan,
            "Spearman ρ": spearmanr(y_true, y_pred)[0] if np.std(y_pred) > 0 else np.nan,
            "n_test": len(y_true)}

qsar = agg[agg.gene == QSAR_TARGET].reset_index(drop=True)
X_q, fps_q = fingerprints(qsar.smiles)
y_q = qsar.pchembl.values
print(f"{QSAR_TARGET}: {X_q.shape[0]} compounds x {X_q.shape[1]} bits")

### 3.2 Three ways to split the data

* **Random split** — the test set contains close analogues of training compounds (same paper, same series). Optimistic.
* **Scaffold split** — all compounds sharing a Bemis–Murcko scaffold go to the same set, so the test set contains *new chemotypes*.
* **Temporal split** — train on compounds published up to year *t*, test on later ones. This mimics prospective use and is usually the most realistic (and most pessimistic).

In [ ]:
def random_split(n, test_frac=0.2, seed=SEED):
    idx_train, idx_test = train_test_split(np.arange(n), test_size=test_frac, random_state=seed)
    return np.sort(idx_train), np.sort(idx_test)

def scaffold_split(smiles, test_frac=0.2):
    """Largest scaffold groups go to training; small/rare scaffolds end up in the test set."""
    scaffolds = pd.Series([MurckoScaffold.MurckoScaffoldSmiles(smiles=s) for s in smiles])
    groups = scaffolds.groupby(scaffolds).groups
    train, cutoff = [], (1 - test_frac) * len(smiles)
    for _, idx in sorted(groups.items(), key=lambda kv: len(kv[1]), reverse=True):
        if len(train) + len(idx) <= cutoff:
            train.extend(idx)
    train = np.sort(np.array(train))
    test = np.setdiff1d(np.arange(len(smiles)), train)
    return train, test

def temporal_split(years, test_frac=0.2):
    years = pd.Series(years, dtype=float).reset_index(drop=True)
    years = years.fillna(years.max())          # unknown year -> treat as most recent
    cutoff = years.quantile(1 - test_frac)
    train = np.where(years < cutoff)[0]
    test = np.where(years >= cutoff)[0]
    return train, test, int(cutoff)

splits = {"random": random_split(len(qsar)), "scaffold": scaffold_split(qsar.smiles)}
tr, te, cutoff_year = temporal_split(qsar.year)
splits[f"temporal (test ≥ {cutoff_year})"] = (tr, te)
for k, (a, b) in splits.items():
    print(f"{k:28s} train={len(a):6d}  test={len(b):6d}")

In [ ]:
def make_rf(**kw):
    return RandomForestRegressor(n_estimators=N_TREES, max_features=0.3, min_samples_leaf=1,
                                 n_jobs=N_CPU, random_state=SEED, **kw)

qsar_results, qsar_preds = [], {}
for name, (tr, te) in splits.items():
    t0 = time.time()
    model = make_rf().fit(X_q[tr], y_q[tr])
    pred = model.predict(X_q[te])
    qsar_preds[name] = (te, pred)
    res = regression_metrics(y_q[te], pred)
    res.update(split=name, model="RF", seconds=round(time.time() - t0, 1))
    qsar_results.append(res)
    # baseline: predict the training mean for every compound
    base = regression_metrics(y_q[te], np.full(len(te), y_q[tr].mean()))
    base.update(split=name, model="mean baseline")
    qsar_results.append(base)

pd.DataFrame(qsar_results).set_index(["split", "model"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)
for ax, (name, (te, pred)) in zip(axes, qsar_preds.items()):
    ax.scatter(y_q[te], pred, s=6, alpha=0.4)
    lims = [3, 11]; ax.plot(lims, lims, "k--", lw=1)
    ax.fill_between(lims, [l - 1 for l in lims], [l + 1 for l in lims], color="grey", alpha=0.15)
    r = regression_metrics(y_q[te], pred)
    ax.set_title(f"{name}\nR²={r['R2']:.2f}  RMSE={r['RMSE']:.2f}")
    ax.set_xlabel("measured pChEMBL"); ax.set_xlim(lims); ax.set_ylim(lims)
axes[0].set_ylabel("predicted pChEMBL")
plt.suptitle(f"{QSAR_TARGET} random forest QSAR (grey band = ±1 log unit)", y=1.03)
plt.tight_layout(); plt.show()

**Observations to discuss**

* Performance drops from random → scaffold → temporal split. Which number would you report to a medicinal chemist who wants to use the model for *new* compound series?
* Note the "regression to the mean": the model under-predicts the most potent and over-predicts the least potent compounds. Random forests average over training values and cannot extrapolate beyond the observed range.

### 3.3 Y-scrambling (randomisation test)

If a model trained on **shuffled** labels performs as well as the real model, the original model learned nothing meaningful (chance correlation). With 2048 descriptors and a few thousand compounds this sanity check is worth doing.

In [ ]:
tr, te = splits["random"]
scr = []
for i in range(3):
    y_shuffled = rng.permutation(y_q[tr])
    m = make_rf().fit(X_q[tr], y_shuffled)
    scr.append(regression_metrics(y_q[te], m.predict(X_q[te]))["R2"])
print("R² with real labels    :", round(qsar_results[0]["R2"], 3))
print("R² with scrambled labels:", np.round(scr, 3))

### 3.4 Applicability domain

A model is only reliable for compounds that resemble its training set. A simple applicability-domain (AD) measure is the **Tanimoto similarity to the nearest training compound**. Let's see how the prediction error depends on it (scaffold split).

In [ ]:
tr, te = splits["scaffold"]
te_idx, pred = qsar_preds["scaffold"]
train_fps = [fps_q[i] for i in tr]
nn_sim = np.array([max(DataStructs.BulkTanimotoSimilarity(fps_q[i], train_fps)) for i in te_idx])
abs_err = np.abs(y_q[te_idx] - pred)

bins = [0, 0.3, 0.4, 0.5, 0.6, 0.7, 1.01]
ad = pd.DataFrame({"nn_sim": nn_sim, "abs_err": abs_err})
ad["similarity bin"] = pd.cut(ad.nn_sim, bins, right=False)
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=ad, x="similarity bin", y="abs_err", ax=ax, color="lightsteelblue", showfliers=False)
for i, (b, grp) in enumerate(ad.groupby("similarity bin", observed=False)):
    ax.text(i, ax.get_ylim()[1] * 0.95, f"n={len(grp)}", ha="center", fontsize=8)
ax.set_ylabel("|error| (log units)"); ax.set_xlabel("Tanimoto similarity to nearest training compound")
ax.set_title("Prediction error vs. distance to training set (scaffold split)")
plt.tight_layout(); plt.show()

### 3.5 Classification: active vs inactive

In virtual screening we often only need to **rank** or **classify** compounds. Using the threshold pChEMBL ≥ 6.5 we train a random-forest classifier and evaluate it with ROC-AUC, Matthews correlation coefficient (MCC) and balanced accuracy — metrics that are robust to class imbalance (plain accuracy is not).

In [ ]:
yc = (y_q >= ACTIVE_THRESHOLD).astype(int)
clf_res = []
for name, (tr, te) in splits.items():
    clf = RandomForestClassifier(n_estimators=N_TREES, max_features="sqrt", class_weight="balanced",
                                 n_jobs=N_CPU, random_state=SEED).fit(X_q[tr], yc[tr])
    proba = clf.predict_proba(X_q[te])[:, 1]
    clf_res.append({"split": name, "frac_active_test": yc[te].mean(),
                    "ROC-AUC": roc_auc_score(yc[te], proba),
                    "MCC": matthews_corrcoef(yc[te], proba >= 0.5),
                    "balanced acc.": balanced_accuracy_score(yc[te], proba >= 0.5)})
pd.DataFrame(clf_res).set_index("split").round(3)

**Exercise 5.1** — Descriptor choice. Re-run the scaffold-split regression with (a) Morgan radius 3, (b) 1024 bits, (c) a set of ~200 RDKit 2D descriptors (`from rdkit.Chem import Descriptors; Descriptors.CalcMolDescriptors(mol)`, RDKit ≥ 2023.03). Which works best? Is the difference larger than the variation you get from changing the random seed?

<details><summary><b>Solution sketch</b></summary>

```python
tr, te = splits["scaffold"]
for radius, bits in [(2, 2048), (3, 2048), (2, 1024)]:
    X, _ = fingerprints(qsar.smiles, radius=radius, n_bits=bits)
    m = make_rf().fit(X[tr], y_q[tr])
    print(radius, bits, regression_metrics(y_q[te], m.predict(X[te])))

from rdkit.Chem import Descriptors
D = pd.DataFrame([Descriptors.CalcMolDescriptors(Chem.MolFromSmiles(s)) for s in qsar.smiles])
D = D.replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)
m = make_rf().fit(D[tr], y_q[tr])
print("RDKit 2D", regression_metrics(y_q[te], m.predict(D[te])))
```
Differences between reasonable fingerprints are usually small (a few hundredths in R²), often comparable to seed-to-seed variation — data and split matter more than the descriptor.
</details>

In [ ]:
# Your answer to Exercise 5.1 here

**Exercise 5.2** — A *k*-nearest-neighbour model (predict the mean pChEMBL of the 5 most similar training compounds) is a strong and interpretable baseline. Implement it with Tanimoto similarity and compare it with the random forest on the three splits.

<details><summary><b>Solution sketch</b></summary>

```python
def knn_predict(train_idx, test_idx, k=5):
    tfps = [fps_q[i] for i in train_idx]
    preds = []
    for i in test_idx:
        sims = np.array(DataStructs.BulkTanimotoSimilarity(fps_q[i], tfps))
        nn = np.argsort(-sims)[:k]
        preds.append(y_q[train_idx][nn].mean())
    return np.array(preds)

for name, (tr, te) in splits.items():
    print(name, regression_metrics(y_q[te], knn_predict(tr, te)))
```
</details>

In [ ]:
# Your answer to Exercise 5.2 here

## 4. Summary and outlook

* Curation and the **validation design** dominate the reported performance of QSAR models; always report a scaffold or temporal split next to a random split, and compare against simple baselines (mean, kNN).
* Random-split performance is optimistic; the applicability domain tells you for which compounds a prediction can be trusted.
* A QSAR model covers **one** target. In **Notebook 6** we extend this to **proteochemometrics (PCM)**: one model for 12 kinases at once, using a description of the KLIFS binding pocket, and we test whether it can predict a kinase it has never seen.